# Milestone 2 — stem-grouped CV split

In [2]:
# imports
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupKFold
sys.path.insert(0, str(Path("..").resolve()))
from src.normalize import normalize_stem

In [3]:
# load train and attach normalized stem
data_dir = Path("../data")
train = pd.read_csv(data_dir / "train.csv")
train["stem"] = train["prompt"].apply(normalize_stem)
print(train.shape, "unique stems:", train["stem"].nunique())

(2000, 9) unique stems: 252


In [4]:
# stem-grouped 5-fold split
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
train["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(gkf.split(train, groups=train["stem"])):
    train.loc[val_idx, "fold"] = fold_idx
assert (train["fold"] >= 0).all(), "some rows missed fold assignment"
print(train["fold"].value_counts().sort_index())

fold
0    400
1    400
2    400
3    400
4    400
Name: count, dtype: int64


In [5]:
# verify stem disjointness across folds
for k in range(N_FOLDS):
    val_stems = set(train.loc[train["fold"] == k, "stem"])
    train_stems = set(train.loc[train["fold"] != k, "stem"])
    overlap = val_stems & train_stems
    print(f"fold {k}: val_stems={len(val_stems)}, train_stems={len(train_stems)}, overlap={len(overlap)}")
    assert len(overlap) == 0, f"stem overlap in fold {k}"

fold 0: val_stems=50, train_stems=202, overlap=0
fold 1: val_stems=50, train_stems=202, overlap=0
fold 2: val_stems=50, train_stems=202, overlap=0
fold 3: val_stems=51, train_stems=201, overlap=0
fold 4: val_stems=51, train_stems=201, overlap=0


In [6]:
# answer label distribution per fold
print(pd.crosstab(train["fold"], train["answer"], normalize="index").round(4))

answer       A       B       C       D       E
fold                                          
0       0.1800  0.2275  0.2525  0.2250  0.1150
1       0.2200  0.2150  0.2450  0.1950  0.1250
2       0.1675  0.1950  0.2950  0.1525  0.1900
3       0.2000  0.3325  0.1875  0.2075  0.0725
4       0.1550  0.2550  0.1675  0.1150  0.3075


In [7]:
# save fold assignment to src/cv_folds.csv
out_path = Path("../src/cv_folds.csv")
train[["id", "fold"]].to_csv(out_path, index=False)
print(f"wrote {len(train)} rows to {out_path}")
print(pd.read_csv(out_path).head())

wrote 2000 rows to ..\src\cv_folds.csv
   id  fold
0   1     2
1   2     1
2   3     2
3   4     2
4   5     1
